# Практикум: виртуальное окружение и Git

Python, 9 класс (углублённый). Раздел 3.1 (окончание) и раздел 3.2, пара 1.

Двенадцать задач тремя уровнями:

- **Уровень 1 (задачи 1–6)** — обязательный. По одной идее на задачу.
- **Уровень 2 (задачи 7–10)** — на сочетание идей.
- **Уровень 3 (задачи 11–12)** — со звёздочкой.

Почти все задачи — про проект «Дневник погоды»: модуль `weather.py` читает из файла `days.txt` записи
о погоде (дата, температура, что на небе) и считает простую статистику.

**Как работать.** У каждой задачи своя папка. Ячейка задачи начинается с `start(N)`: эта команда готовит
папку задачи заново — файлы и, если нужно, историю коммитов. Ниже, на месте `# ваш код здесь`, пишите
команды, как в лекции: `git("status")`, `git('commit -m "..."')`, `run(...)`, `venv_run(...)`. Ещё
пригодятся `write(путь, текст)`, `read(путь)`, `show_tree()` и `git_output("...")` — она возвращает
вывод команды Git строкой и ничего не печатает. Ниже строки `# ---------- проверка ----------` — тесты:
они смотрят, что получилось в папке и в репозитории. Если все проверки проходят, внизу печатается «ок».

Если запутались, просто запустите ячейку ещё раз: `start` вернёт всё в исходное состояние. Перед
решением полезно посмотреть, что в папке: `git("status")` и `git("log --oneline")`.

В задачах 7, 9, 11 и 12 нужно написать функцию. Задаче 1 нужен интернет: она ставит библиотеку из PyPI.
Первую ячейку с инструментами нужно запустить.

In [ ]:
# готовые инструменты для задач и проверок — писать их не нужно, просто запустите ячейку
import ast
import contextlib
import os
import re
import shlex
import shutil
import stat
import subprocess
import sys
import tempfile
import textwrap
from datetime import datetime, timedelta, timezone
from pathlib import Path

sys.dont_write_bytecode = True        # чтобы после правки файла Python не взял его старую версию
_BASE = Path(tempfile.gettempdir()) / "py9_17_practice"
_BASE.mkdir(parents=True, exist_ok=True)
ROOT = _BASE / "task"
_WINDOWS = os.name == "nt"
_START = datetime(2026, 11, 9, 10, 0, tzinfo=timezone(timedelta(hours=3)))
_ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")
_ENV = {name: value for name, value in os.environ.items()
        if not name.startswith(("GIT_", "PIP_"))
        and name not in ("VIRTUAL_ENV", "FORCE_COLOR", "CLICOLOR_FORCE", "PY_COLORS")}
_ENV.update({
    "PYTHONDONTWRITEBYTECODE": "1", "PYTHONIOENCODING": "utf-8", "PYTHON_COLORS": "0",
    "NO_COLOR": "1", "TERM": "dumb", "LANGUAGE": "en_US:en",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1", "PIP_NO_INPUT": "1", "PIP_ROOT_USER_ACTION": "ignore",
    "PIP_PROGRESS_BAR": "off",
    # общие настройки Git — во временном файле, ваши настоящие настройки не меняются
    "GIT_CONFIG_GLOBAL": str(_BASE / "gitconfig"), "GIT_CONFIG_NOSYSTEM": "1",
    "GIT_CEILING_DIRECTORIES": str(_BASE), "GIT_PAGER": "cat", "PAGER": "cat",
    "GIT_EDITOR": "true", "GIT_TERMINAL_PROMPT": "0",
})


# ---------------------------------------------------------------- файлы и команды

def _rmtree(path):
    def retry(func, name, *_):
        os.chmod(name, stat.S_IWRITE)         # в Windows файлы внутри .git только для чтения
        func(name)
    if sys.version_info >= (3, 12):
        shutil.rmtree(path, onexc=retry)
    else:
        shutil.rmtree(path, onerror=retry)


def _write(path, text):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(text).lstrip("\n"), encoding="utf-8", newline="\n")


def write(path, text):
    """Записывает файл в папке задачи (остальные файлы не трогает)."""
    _write(ROOT / path, text)


def read(path):
    """Текст файла из папки задачи."""
    return (ROOT / path).read_text(encoding="utf-8")


def remove(path):
    """Удаляет файл или папку в папке задачи."""
    target = ROOT / path
    if target.is_dir() and not target.is_symlink():
        _rmtree(target)
    elif target.exists() or target.is_symlink():
        target.unlink()


def show_tree(path=".", depth=None):
    """Показывает файлы и папки задачи; depth — сколько уровней. Содержимое .git не показывается."""
    start = ROOT / path
    print(f"{ROOT.name}/" if path == "." else f"{path}/")

    def walk(folder, level):
        for item in sorted(folder.iterdir(), key=lambda item: item.name):
            folder_like = item.is_dir() and not item.is_symlink()
            print("    " * level + item.name + ("/" if item.is_dir() else ""))
            if folder_like and item.name != ".git" and (depth is None or level < depth):
                walk(item, level + 1)

    walk(start, 1)


def _execute(command, env=None):
    result = subprocess.run(command, cwd=ROOT, env=env or _ENV, capture_output=True, text=True,
                            encoding="utf-8", errors="replace", stdin=subprocess.DEVNULL, timeout=600)
    return result.returncode, _ANSI.sub("", result.stdout), _ANSI.sub("", result.stderr).strip()


def _report(code, out, error):
    print(out, end="")
    if code != 0 and "Traceback" in error:    # от трассировки оставляем последнюю строку
        error = error.splitlines()[-1]
    if error:
        print("[ошибка]" if code != 0 else "", error)


def _make_venv(args):
    # python -m venv ...; если в системе нет ensurepip (так бывает в Colab), окружение делает virtualenv
    code, out, error = _execute([sys.executable, *args])
    if code != 0 and "ensurepip" in error:
        for name in args[2:]:
            if not name.startswith("-") and (ROOT / name).exists():
                _rmtree(ROOT / name)
        _execute([sys.executable, "-m", "pip", "install", "-q", "virtualenv"])
        options = ["--no-pip" if arg == "--without-pip" else arg for arg in args[2:]]
        code, out, error = _execute([sys.executable, "-m", "virtualenv", "-q", *options])
        if code == 0:
            out += ("(в этой системе у venv нет модуля ensurepip, поэтому окружение "
                    "создано пакетом virtualenv — результат тот же)\n")
        else:
            error = ("в этой системе у venv нет модуля ensurepip, а поставить virtualenv "
                     "не получилось — проверьте интернет")
    return code, out, error


def run(*args):
    """Запускает в папке задачи  python args...  — тот Python, на котором работает ноутбук."""
    print("$ python", *args)
    if args[:2] == ("-m", "venv"):
        _report(*_make_venv(args))
    else:
        _report(*_execute([sys.executable, *args]))


def _venv_python(venv):
    return ROOT / venv / ("Scripts/python.exe" if _WINDOWS else "bin/python")


def venv_run(*args, venv=".venv"):
    """Запускает в папке задачи Python виртуального окружения (по умолчанию из папки .venv)."""
    print("$", f"{venv}\\Scripts\\python" if _WINDOWS else f"{venv}/bin/python", *args)
    python = _venv_python(venv)
    if not python.exists():
        print(f"[ошибка] окружения {venv} нет, сначала создайте его: run('-m', 'venv', '{venv}')")
        return
    env = {name: value for name, value in _ENV.items() if name not in ("PYTHONPATH", "PYTHONHOME")}
    _report(*_execute([str(python), *args], env))


def _venv_output(*args, venv=".venv"):
    env = {name: value for name, value in _ENV.items() if name not in ("PYTHONPATH", "PYTHONHOME")}
    return _execute([str(_venv_python(venv)), *args], env)[1]


def _git(args, env=None):
    return subprocess.run(["git", "-c", "core.quotePath=false", "-c", "color.ui=never", *args],
                          cwd=ROOT, env=env or _ENV, capture_output=True, stdin=subprocess.DEVNULL,
                          text=True, encoding="utf-8", errors="replace", timeout=60)


def _dated_env():
    # время коммита зависит только от их числа: так история задачи одинакова у всех
    count = _git(["rev-list", "--count", "HEAD"]).stdout.strip()
    moment = _START + timedelta(minutes=5 * (int(count) if count.isdigit() else 0))
    stamp = f"{int(moment.timestamp())} +0300"
    return {**_ENV, "GIT_AUTHOR_DATE": stamp, "GIT_COMMITTER_DATE": stamp}


def git(command):
    """Выполняет в папке задачи команду  git command  и печатает, что она вывела."""
    print("$ git", command)
    result = subprocess.run(["git", "-c", "core.quotePath=false", "-c", "color.ui=never",
                             *shlex.split(command)], cwd=ROOT, env=_dated_env(),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                            text=True, encoding="utf-8", errors="replace", timeout=60)
    text = _ANSI.sub("", result.stdout).rstrip()
    if text:
        print(text)


def git_output(command):
    """Выполняет  git command  и возвращает то, что она напечатала, строкой (ничего не печатает)."""
    result = _git(shlex.split(command), _dated_env())
    if result.returncode != 0:
        raise RuntimeError(f"git {command}: {result.stderr.strip()}")
    return result.stdout


# ---------------------------------------------------------------- для проверок

def log():
    """История текущей ветки от старых коммитов к новым: список пар (хеш, сообщение)."""
    result = _git(["log", "--reverse", "--format=%H%x09%s"])
    if result.returncode != 0:
        return []
    return [tuple(line.split("\t", 1)) for line in result.stdout.splitlines()]


def changed(commit):
    """Множество файлов, которые изменил коммит."""
    out = _git(["diff-tree", "--no-commit-id", "--name-only", "-r", "--root", commit]).stdout
    return set(out.split("\n")) - {""}


def file_at(commit, path):
    """Текст файла в коммите или None, если файла там нет."""
    result = _git(["show", f"{commit}:{path}"])
    return result.stdout if result.returncode == 0 else None


def status():
    """Короткий git status: список строк вида ' M файл', '?? файл'."""
    return [line for line in _git(["status", "--porcelain"]).stdout.split("\n") if line]


def tracked():
    """Множество файлов, которые лежат в репозитории (в индексе)."""
    return set(_git(["ls-files"]).stdout.split("\n")) - {""}


def ignored(paths):
    """Какие из путей (их может и не быть на диске) Git игнорирует по .gitignore задачи."""
    result = subprocess.run(["git", "check-ignore", "--no-index", "--stdin"], cwd=ROOT, env=_ENV,
                            input="\n".join(paths) + "\n", capture_output=True, text=True,
                            encoding="utf-8")
    return set(result.stdout.split("\n")) - {""}


def venv_has(venv=".venv"):
    """Есть ли в папке задачи окружение venv."""
    return _venv_python(venv).exists()


def venv_freeze(venv=".venv"):
    """Что печатает pip freeze в окружении: список строк."""
    return [line for line in _venv_output("-m", "pip", "freeze", venv=venv).splitlines() if line]


def prepared(path):
    """Текст файла в том виде, в каком его оставил start."""
    return _PREPARED[ROOT.name][path]


def git_hash(content):
    """Хеш, который Git дал бы файлу с содержимым content (bytes), — git hash-object."""
    result = subprocess.run(["git", "hash-object", "--stdin"], input=content, capture_output=True,
                            cwd=_BASE, env=_ENV)
    return result.stdout.decode().strip()


def git_ignores(patterns, paths):
    """Какие из путей игнорирует настоящий Git, если в .gitignore записаны строки patterns."""
    probe = _BASE / "ignore_probe"
    if not (probe / ".git").exists():
        probe.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "init", "-q"], cwd=probe, env=_ENV, check=True)
    (probe / ".gitignore").write_text("\n".join(patterns) + "\n", encoding="utf-8", newline="\n")
    result = subprocess.run(["git", "check-ignore", "--no-index", "--stdin"], cwd=probe, env=_ENV,
                            input="\n".join(paths) + "\n", capture_output=True, text=True, encoding="utf-8")
    return set(result.stdout.split("\n")) - {""}


def module_from(text):
    """Выполняет текст модуля и возвращает его переменные словарём."""
    namespace = {"__name__": "checked_module"}
    exec(text, namespace)
    return namespace


def error_of(func, *args, **kwargs):
    """Вызывает func и возвращает исключение, если оно случилось, иначе None."""
    try:
        func(*args, **kwargs)
    except Exception as error:
        return error
    return None


@contextlib.contextmanager
def no_programs():
    """Внутри блока нельзя запускать программы, в том числе git (для задачи 11)."""
    def forbidden(*args, **kwargs):
        raise RuntimeError("в этой задаче git запускать нельзя: читайте файлы в папке .git")
    space = globals()
    saved = (subprocess.run, subprocess.Popen, os.system, os.popen, space["git"], space["git_output"])
    subprocess.run = subprocess.Popen = os.system = os.popen = forbidden
    space["git"] = space["git_output"] = forbidden
    try:
        yield
    finally:
        subprocess.run, subprocess.Popen, os.system, os.popen, space["git"], space["git_output"] = saved


# ---------------------------------------------------------------- проект «Дневник погоды»

_READ_DAYS = '''
def read_days(path):
    """Читает записи: в каждой строке дата, температура и что на небе."""
    days = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        date, temp, sky = line.split()
        days.append((date, int(temp), sky))
    return days
'''

_AVERAGE = '''
def average_temp(days):
    """Средняя температура."""
    total = 0
    for date, temp, sky in days:
        total += temp
    return total / len(days)
'''

_AVERAGE_EMPTY = '''
def average_temp(days):
    """Средняя температура; None, если записей нет."""
    if not days:
        return None
    total = 0
    for date, temp, sky in days:
        total += temp
    return total / len(days)
'''

_AVERAGE_BAD = '''
def average_temp(days):
    """Средняя температура."""
    return sum(temp for date, temp, sky in days) // len(days)
'''

_RAINY = '''
def rainy_days(days):
    """Даты дождливых дней."""
    return [date for date, temp, sky in days if sky == "дождь"]
'''

_WARMEST = '''
def warmest_day(days):
    """Дата самого тёплого дня."""
    best = days[0]
    for day in days:
        if day[1] > best[1]:
            best = day
    return best[0]
'''

_RANGE = '''
def temp_range(days):
    """Разница между самой высокой и самой низкой температурой."""
    temps = [temp for date, temp, sky in days]
    return max(temps) - min(temps)
'''


def _weather(*functions):
    head = '"""Дневник погоды: чтение записей и простая статистика."""\n\nfrom pathlib import Path\n'
    return head + "".join("\n\n" + textwrap.dedent(function).strip("\n") + "\n" for function in functions)


_DAYS_1 = "2026-11-01 6 дождь\n2026-11-02 4 облачно\n2026-11-03 2 снег\n2026-11-04 5 дождь\n2026-11-05 7 солнце\n"
_DAYS_2 = _DAYS_1 + "2026-11-06 3 облачно\n2026-11-07 1 снег\n2026-11-08 4 дождь\n"
_DAYS_3 = _DAYS_2 + "2026-11-09 0 снег\n2026-11-10 -2 солнце\n"
_README = "# Дневник погоды\n\nЗаписи о погоде по дням: дата, температура, что на небе.\n"
_README_USAGE = _README + (
    "\nКак пользоваться:\n\n```python\nfrom weather import read_days, average_temp\n\n"
    "print(average_temp(read_days(\"days.txt\")))\n```\n")
_README_ARCHIVE = _README + "\nЗаписи за октябрь — в файле `archive_october.txt`.\n"
_ARCHIVE = "2026-10-28 9 дождь\n2026-10-29 8 облачно\n2026-10-30 10 солнце\n2026-10-31 7 дождь\n"
_FIRST = {"weather.py": _weather(_READ_DAYS, _AVERAGE), "days.txt": _DAYS_1, "README.md": _README}


def _weather_9(number, bad_at, bad_body, crash_body):
    """weather.py в коммите number задачи 9: average_temp портится в коммите bad_at."""
    docs, typed = number >= 9, number >= 11
    if number >= bad_at:
        body = crash_body if crash_body and number < 7 else bad_body
    elif number >= 7:
        body = "return sum(temp for _, temp, _ in days) / len(days)"
    elif number >= 5:
        body = "result = 0\nfor _, t, _ in days:\n    result += t\nreturn result / len(days)"
    else:
        body = "total = 0\nfor date, temp, sky in days:\n    total += temp\nreturn total / len(days)"

    days_type = "list[tuple[str, int, str]]"

    def function(name, param, param_type, result_type, doc, code):
        head = f"def {name}({param}: {param_type}) -> {result_type}:" if typed else f"def {name}({param}):"
        lines = [head] + ([f'    """{doc}"""'] if docs else [])
        return "\n".join(lines + ["    " + line for line in code.split("\n")])

    parts = [
        function("read_days", "path", "str", days_type, "Читает записи из файла.",
                 'days = []\nfor line in Path(path).read_text(encoding="utf-8").splitlines():\n'
                 '    date, temp, sky = line.split()\n    days.append((date, int(temp), sky))\nreturn days'),
        function("average_temp", "days", days_type, "float", "Средняя температура.", body),
    ]
    if number >= 2:
        parts.append(function("rainy_days", "days", days_type, "list[str]", "Даты дождливых дней.",
                              'return [date for date, temp, sky in days if sky == "дождь"]'))
    if number >= 4:
        parts.append(function("warmest_day", "days", days_type, "str", "Дата самого тёплого дня.",
                              "return max(days, key=lambda day: day[1])[0]"))
    if number >= 8:
        parts.append(function("temp_range", "days", days_type, "int", "Размах температур.",
                              "temps = [temp for _, temp, _ in days]\nreturn max(temps) - min(temps)"))
    return '"""Дневник погоды."""\n\nfrom pathlib import Path\n\n\n' + "\n\n\n".join(parts) + "\n"


_MESSAGES_9 = ["Первая версия дневника погоды", "Дождливые дни", "Записи за первую неделю", "Самый тёплый день",
               "Переименовать переменные в average_temp", "Записи за вторую неделю", "Ускорить подсчёт средней",
               "Размах температур", "Докстроки к функциям", "Записи за третью неделю", "Аннотации типов",
               "Записи за четвёртую неделю"]
_VARIANTS_9 = {   # номер коммита, с которого average_temp сломана; ошибка; ошибка, из-за которой функция падает
    0: (7, "return sum(temp for _, temp, _ in days) // len(days)", None),
    1: (5, "return sum(temp for _, temp, _ in days) // len(days)",
        "result = 0\nfor _, t, _ in days:\n    result += t\nreturn reslt / len(days)"),
    2: (11, "return round(sum(temp for _, temp, _ in days) / len(days))", None),
}


def _history_9(variant):
    bad_at, bad_body, crash_body = _VARIANTS_9[variant]
    weeks = {1: "".join(_DAYS_1.splitlines(True)[:3]), 3: _DAYS_1, 6: _DAYS_2, 10: _DAYS_3,
             12: _DAYS_3 + "2026-11-11 -1 снег\n2026-11-12 1 облачно\n"}
    steps, days = [], weeks[1]
    for number, message in enumerate(_MESSAGES_9, start=1):
        days = weeks.get(number, days)
        files = {"weather.py": _weather_9(number, bad_at, bad_body, crash_body), "days.txt": days}
        if number == 1:
            files["README.md"] = _README
        steps.append(("commit", message, files))
    return steps


def _history_11(variant):
    if variant == 0:
        return [("commit", "Первая версия дневника погоды", _FIRST),
                ("commit", "Дождливые дни\n\nФункция rainy_days возвращает даты.",
                 {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY)}),
                ("commit", "Записи за 6–8 ноября", {"days.txt": _DAYS_2}),
                ("commit", "Самый тёплый день", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY, _WARMEST)})]
    return [("commit", "Start", {"notes.txt": "first\n"}),
            ("commit", "Second line\n\nlonger description\non two lines", {"notes.txt": "first\nsecond\n"}),
            ("commit", "Пустой файл", {"empty.txt": ""}),
            ("commit", "Удалить пустой файл", {"empty.txt": None}),
            ("commit", "Ёлка 🎄", {"tree.txt": "*\n***\n"}),
            ("commit", "Финал", {"notes.txt": "first\nsecond\nthird\n"})]


_TASKS = {
    1: {"files": {
        "requirements.txt": "tabulate==0.8.10\n",
        "table.py": ('from tabulate import __version__, tabulate\n\nprint("tabulate", __version__)\n'
                     'print(tabulate([["Москва", 3], ["Казань", 1]], headers=["Город", "Температура"]))\n'),
    }},
    2: {"files": {**_FIRST, "draft.txt": "черновик: добавить ветер? спросить у Пети, где брать прогноз\n"}},
    3: {"steps": [("commit", "Первая версия дневника погоды", _FIRST),
                  ("write", {"weather.py": _weather(_READ_DAYS, _AVERAGE_EMPTY), "README.md": _README_USAGE}),
                  ("stage", ["weather.py", "README.md"])]},
    4: {"steps": [("commit", "Первая версия дневника погоды", _FIRST),
                  ("commit", "Дождливые дни", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY)}),
                  ("write", {"weather.py": _weather(_READ_DAYS.replace("int(temp)", "temp"), _AVERAGE, _RAINY)}),
                  ("stage", ["weather.py"]),
                  ("write", {"days.txt": _DAYS_2})]},
    5: {"init": True, "files": {
        "main.py": ('from weather.stats import average_temp\nfrom weather.storage import read_days\n\n'
                    'print(average_temp(read_days("data/november.txt")))\n'),
        "weather/__init__.py": '"""Дневник погоды."""\n',
        "weather/stats.py": textwrap.dedent(_AVERAGE).lstrip("\n"),
        "weather/storage.py": "from pathlib import Path\n\n\n" + textwrap.dedent(_READ_DAYS).lstrip("\n"),
        "data/november.txt": _DAYS_2,
        "requirements.txt": "requests==2.32.3\n",
        "analysis.ipynb": '{"cells": [], "metadata": {}, "nbformat": 4, "nbformat_minor": 5}\n',
        "api_key.txt": "ключ-к-сайту-с-прогнозом-никому-не-показывать\n",
        "weather.log": "2026-11-08 10:00 прочитано 8 записей\n",
        ".venv/pyvenv.cfg": "home = /usr/bin\ninclude-system-site-packages = false\n",
        ".venv/lib/python3.12/site-packages/requests/__init__.py": "# библиотека requests\n",
        "__pycache__/main.cpython-312.pyc": "байт-код\n",
        "weather/__pycache__/stats.cpython-312.pyc": "байт-код\n",
        ".ipynb_checkpoints/analysis-checkpoint.ipynb": "{}\n",
    }},
    6: {"steps": [("commit", "Первая версия дневника погоды", _FIRST),
                  ("commit", "Самый тёплый день", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _WARMEST)}),
                  ("commit", "Переписать подсчёт средней",
                   {"weather.py": _weather(_READ_DAYS, _AVERAGE_BAD, _WARMEST), "days.txt": _DAYS_2}),
                  ("commit", "Записи за 9–10 ноября", {"days.txt": _DAYS_3})]},
    8: {"steps": [("commit", "Первая версия дневника погоды", _FIRST),
                  ("commit", "Самый тёплый день",
                   {"weather.py": _weather(_READ_DAYS, _AVERAGE, _WARMEST.replace(">", "<"))}),
                  ("write", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _WARMEST, _RANGE)})]},
    10: {"steps": [("commit", "Первая версия дневника погоды", _FIRST),
                   ("commit", "Архив за октябрь", {"archive_october.txt": _ARCHIVE, "README.md": _README_ARCHIVE}),
                   ("commit", "Дождливые дни", {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY)}),
                   ("commit", "Убрать архив", {"archive_october.txt": None, "README.md": _README}),
                   ("commit", "Записи за 6–8 ноября", {"days.txt": _DAYS_2}),
                   ("commit", "Самый тёплый день",
                    {"weather.py": _weather(_READ_DAYS, _AVERAGE, _RAINY, _WARMEST)})]},
}


def _quiet_git(*args, dated=False):
    result = _git(list(args), _dated_env() if dated else _ENV)
    if result.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)}: {result.stderr.strip()}")


def _put(files):
    for name, text in files.items():
        if text is None:
            remove(name)
        else:
            _write(ROOT / name, text)


def start(task, variant=0, quiet=False):
    """Готовит папку задачи task заново: файлы и, если нужно, историю Git."""
    global ROOT
    ROOT = _BASE / f"task_{task}"
    if ROOT.exists():
        _rmtree(ROOT)
    ROOT.mkdir(parents=True)
    spec = _TASKS.get(task, {})
    if task == 9:
        spec = {"steps": _history_9(variant)}
    if task == 11:
        spec = {"steps": _history_11(variant)}
    _put(spec.get("files", {}))
    if spec.get("init") or spec.get("steps"):
        _quiet_git("init", "-q", "-b", "main")
        _quiet_git("config", "user.name", "Ученик 9С")
        _quiet_git("config", "user.email", "student@example.com")
    for kind, *rest in spec.get("steps", []):
        if kind == "commit":
            message, files = rest
            _put(files)
            _quiet_git("add", "-A")
            _quiet_git("commit", "-q", "-m", message, dated=True)
        elif kind == "write":
            _put(rest[0])
        elif kind == "stage":
            _quiet_git("add", *rest[0])
    _SETUP[task] = log()
    _PREPARED[ROOT.name] = {path.relative_to(ROOT).as_posix(): path.read_text(encoding="utf-8")
                            for path in ROOT.rglob("*")
                            if path.is_file() and ".git" not in path.relative_to(ROOT).parts}
    if not quiet:
        print(f"— папка задачи {task} готова —")


_SETUP = {}
_PREPARED = {}


def setup_log():
    """История, которую приготовил start для текущей задачи: пары (хеш, сообщение) от старых к новым."""
    return _SETUP[int(ROOT.name.split("_")[1])]


if shutil.which("git") is None:
    print("Внимание: Git не найден. Установите его с https://git-scm.com — без него задачи 2–6, 8–11 не работают.")
print("инструменты загружены")

инструменты загружены


## Уровень 1. Обязательный

### Задача 1. Окружение по списку

В папке задачи лежат программа `table.py` и файл `requirements.txt`, в котором записано, какая
библиотека ей нужна. Создайте в папке задачи виртуальное окружение `.venv` и поставьте в него ровно то,
что перечислено в `requirements.txt`, с теми же версиями. Потом запустите `table.py` Python'ом окружения:
программа напечатает версию библиотеки и маленькую таблицу.

*Подсказка:* блок 2 лекции. Имя библиотеки руками не пишите: для установки по списку у `pip` есть ключ
`-r`. Проверка спросит `pip freeze` окружения, что в нём стоит.

In [ ]:
start(1)
run('-m', 'venv', '.venv')
venv_run("-m", "pip", "install", "-q", "-r", "requirements.txt")


# ---------- проверка ----------
assert venv_has(".venv"), "в папке задачи нет окружения .venv"
frozen = venv_freeze(".venv")
assert "tabulate==0.8.10" in frozen, f"в окружении должна стоять tabulate 0.8.10, а pip freeze показывает: {frozen}"
assert frozen == ["tabulate==0.8.10"], f"в окружении должно быть только то, что в requirements.txt, а там: {frozen}"
print("Задача 1: ок")

— папка задачи 1 готова —
$ python -m venv .venv
(в этой системе у venv нет модуля ensurepip, поэтому окружение создано пакетом virtualenv — результат тот же)
$ .venv/bin/python -m pip install -q -r requirements.txt
Задача 1: ок


### Задача 2. Первый коммит

В папке задачи лежит проект без репозитория: `weather.py`, `days.txt`, `README.md` и `draft.txt` — личные
заметки, которым в репозитории не место. Создайте репозиторий с основной веткой `main` и сделайте
первый коммит с сообщением `Первая версия дневника погоды`. В коммит должны войти три файла проекта,
а `draft.txt` — нет. Файл `.gitignore` в этой задаче не нужен.

*Подсказка:* блок 3 лекции. Если Git ответит «Author identity unknown», он просит представиться:
выполните две команды `git config --global ...` из лекции со своим именем.

In [ ]:
start(2)
git('init -b main')
git('config --global user.email "abc123@gmail.com"')
git('config --global user.name "Имя Фамилия"')
git('add weather.py days.txt README.md')
git("commit -m 'Первая версия дневника погоды'")


# ---------- проверка ----------
assert (ROOT / ".git").is_dir(), "репозитория нет: git init -b main"
branch = git_output("branch --show-current").strip()
assert branch == "main", f"основная ветка должна называться main, а называется {branch!r}"
history = log()
assert len(history) == 1, f"нужен ровно один коммит, а их {len(history)}"
assert history[0][1] == "Первая версия дневника погоды", f"сообщение коммита: {history[0][1]!r}"
assert tracked() == {"weather.py", "days.txt", "README.md"}, f"в репозитории лежат: {sorted(tracked())}"
assert status() == ["?? draft.txt"], f"git status должен показывать только неотслеживаемый draft.txt, а показывает: {status()}"
print("Задача 2: ок")

— папка задачи 2 готова —
$ git init -b main
Initialized empty Git repository in /tmp/py9_17_practice/task_2/.git/
$ git config --global user.email "abc123@gmail.com"
$ git config --global user.name "Имя Фамилия"
$ git add weather.py days.txt README.md
$ git commit -m 'Первая версия дневника погоды'
[main (root-commit) 32bb101] Первая версия дневника погоды
 3 files changed, 28 insertions(+)
 create mode 100644 README.md
 create mode 100644 days.txt
 create mode 100644 weather.py
Задача 2: ок


### Задача 3. Каждой правке — свой коммит

Вася поправил в проекте два файла. В `weather.py` функция `average_temp` теперь возвращает `None` для
пустого дневника, а не падает. В `README.md` появился раздел «Как пользоваться». Обе правки он сразу
положил в индекс командой `git add .`. Изменения разные по смыслу, поэтому нужны два коммита: сначала
`weather.py` с сообщением `Средняя температура пустого дневника`, потом `README.md` с сообщением
`Как пользоваться модулем`.

*Подсказка:* посмотрите `git status` — оба файла уже в индексе. Как убрать файл из индекса и не потерять
правку, сказано в блоке 5 лекции.

In [ ]:
start(3)
git('restore --staged README.md')
git('commit -m "Средняя температура пустого дневника"')
git('add README.md')
git('commit -m "Как пользоваться модулем"')




# ---------- проверка ----------
history = log()
assert history[:1] == setup_log(), "старый коммит должен остаться как был"
assert len(history) == 3, f"нужно ровно два новых коммита, а их {len(history) - 1}"
assert history[1][1] == "Средняя температура пустого дневника", f"сообщение первого нового коммита: {history[1][1]!r}"
assert history[2][1] == "Как пользоваться модулем", f"сообщение второго нового коммита: {history[2][1]!r}"
assert changed("HEAD~1") == {"weather.py"}, f"первый новый коммит должен менять только weather.py, а меняет: {sorted(changed('HEAD~1'))}"
assert changed("HEAD") == {"README.md"}, f"второй новый коммит должен менять только README.md, а меняет: {sorted(changed('HEAD'))}"
weather = module_from(file_at("HEAD", "weather.py"))
assert weather["average_temp"]([]) is None, "правка Васи в weather.py потерялась"
assert "Как пользоваться" in file_at("HEAD", "README.md"), "правка Васи в README.md потерялась"
assert status() == [], f"всё должно быть закоммичено, а git status показывает: {status()}"
print("Задача 3: ок")

— папка задачи 3 готова —
$ git restore --staged README.md
$ git commit -m "Средняя температура пустого дневника"
[main 2d2ba3a] Средняя температура пустого дневника
 1 file changed, 3 insertions(+), 1 deletion(-)
$ git add README.md
$ git commit -m "Как пользоваться модулем"
[main 208f199] Как пользоваться модулем
 1 file changed, 8 insertions(+)
Задача 3: ок


### Задача 4. Отменить неудачную правку

Кто-то поменял в `weather.py` строку `int(temp)` на `temp` и положил это в индекс. Теперь температуры
читаются как строки, и `average_temp` падает. Уберите эту правку совсем, и из индекса, и из рабочей
папки, чтобы `weather.py` снова совпадал с последним коммитом. Новые записи в `days.txt` нужные: их
не трогайте и пока не коммитьте. Коммитов делать не нужно.

*Подсказка:* посмотрите `git status`: в каком разделе `weather.py`? Обычный `git restore файл` берёт
версию из индекса. Поможет ли он, если плохая версия уже там?

In [ ]:
start(4)
git('status')
git('restore --source=HEAD --staged --worktree weather.py')

# ---------- проверка ----------
assert log() == setup_log(), "коммитов делать не нужно, историю менять тоже"
assert read("weather.py") == file_at("HEAD", "weather.py"), "weather.py в рабочей папке должен совпадать с последним коммитом"
assert status() == [" M days.txt"], f"git status должен показывать только изменённый days.txt, а показывает: {status()}"
assert read("days.txt") == prepared("days.txt"), "новые записи в days.txt потерялись"
print("Задача 4: ок")

— папка задачи 4 готова —
$ git status
On branch main
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   weather.py

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   days.txt
$ git restore --source=HEAD --staged --worktree weather.py
Задача 4: ок


### Задача 5. Лишнее — в .gitignore

В папке задачи — проект дневника погоды (пакет `weather`, `main.py`, записи в `data/`, блокнот
`analysis.ipynb`, `requirements.txt`) и всё, что накопилось за время работы: окружение `.venv`, кеш
байт-кода `__pycache__` в корне и в пакете `weather`, служебная папка Jupyter `.ipynb_checkpoints`, журнал
программы `weather.log` и файл `api_key.txt` с ключом к сайту с прогнозом. Репозиторий уже создан,
коммитов нет.

Напишите `.gitignore` так, чтобы `git status` показывал только файлы проекта, а потом закоммитьте их
вместе с самим `.gitignore` (здесь можно `git add .`). Сообщение — любое осмысленное. Правила должны быть
общими: проверка спросит Git и о файлах, которых пока нет, например о кеше в новом пакете, о файлах
глубоко внутри `.venv` или о журнале `logs/today.log`.

*Подсказка:* блок 6 лекции, таблица шаблонов. Файл удобно записать так:
`write(".gitignore", ''' ... ''')`.

In [ ]:
start(5)
write(".gitignore", '''.venv/
__pycache__/
.ipynb_checkpoints/
*.log
api_key.txt
''')
git('add .')
git('commit -m "Нет лишних файлов"')


# ---------- проверка ----------
project = {".gitignore", "main.py", "weather/__init__.py", "weather/stats.py", "weather/storage.py",
           "data/november.txt", "requirements.txt", "analysis.ipynb"}
assert (ROOT / ".gitignore").exists(), "нет файла .gitignore"
assert tracked() == project, f"в репозитории должны быть файлы проекта и .gitignore, а там: {sorted(tracked())}"
assert len(log()) == 1, f"нужен один коммит, а их {len(log())}"
assert status() == [], f"git status должен быть пустым, а показывает: {status()}"
must_ignore = ["api_key.txt", "weather.log", "logs/today.log", ".venv/bin/python",
               ".venv/lib/python3.12/site-packages/requests/api.py", "plots/__pycache__/draw.cpython-310.pyc",
               "weather/__pycache__/storage.cpython-313.pyc", "notebooks/.ipynb_checkpoints/test-checkpoint.ipynb"]
must_keep = ["main.py", "analysis.ipynb", "requirements.txt", "data/december.txt", "weather/plot.py",
             "notes.md", "logs/README.md"]
answer = ignored(must_ignore + must_keep)
missed = [path for path in must_ignore if path not in answer]
assert not missed, f"эти файлы Git должен игнорировать, а не игнорирует: {missed}"
extra = [path for path in must_keep if path in answer]
assert not extra, f"эти файлы — часть проекта, а Git их игнорирует: {extra}"
print("Задача 5: ок")

— папка задачи 5 готова —
$ git add .
$ git commit -m "Нет лишних файлов"
[main (root-commit) ee4358c] Нет лишних файлов
 8 files changed, 36 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 analysis.ipynb
 create mode 100644 data/november.txt
 create mode 100644 main.py
 create mode 100644 requirements.txt
 create mode 100644 weather/__init__.py
 create mode 100644 weather/stats.py
 create mode 100644 weather/storage.py
Задача 5: ок


### Задача 6. Вернуть старую версию файла

В коммите «Переписать подсчёт средней» функцию `average_temp` попытались ускорить и сломали: теперь
она делит нацело, и средняя для 5 и 4 градусов получается 4, а не 4,5. Верните `weather.py` к версии
из коммита «Самый тёплый день» и закоммитьте с сообщением `Вернуть прежний подсчёт средней`. Записи
в `days.txt`, сделанные после, должны остаться. Историю не переписывайте: старые коммиты должны
остаться на месте.

*Подсказка:* найдите нужный коммит в `git log --oneline` и посмотрите таблицу в конце блока 5 лекции.

In [ ]:
start(6)
git('log --oneline')
git('checkout ec7f1a2 -- weather.py')
git('commit -m "Вернуть прежний подсчёт средней"')

# ---------- проверка ----------
history, original = log(), setup_log()
assert history[:4] == original, "старые коммиты должны остаться как были — историю переписывать нельзя"
assert len(history) > 4, "нового коммита нет. Если git commit ответил «nothing to commit», то взята та же версия weather.py, что уже в последнем коммите"
assert len(history) == 5, f"нужен ровно один новый коммит, а их {len(history) - 4}"
assert history[4][1] == "Вернуть прежний подсчёт средней", f"сообщение нового коммита: {history[4][1]!r}"
good = file_at(original[1][0], "weather.py")
assert file_at("HEAD", "weather.py") == good, "weather.py в новом коммите должен совпадать с версией из «Самый тёплый день»"
assert file_at("HEAD", "days.txt") == file_at(original[3][0], "days.txt"), "записи в days.txt потерялись"
assert changed("HEAD") == {"weather.py"}, f"новый коммит должен менять только weather.py, а меняет: {sorted(changed('HEAD'))}"
assert status() == [], f"всё должно быть закоммичено, а git status показывает: {status()}"
print("Задача 6: ок")

— папка задачи 6 готова —
$ git log --oneline
831fc8f Записи за 9–10 ноября
4f57bac Переписать подсчёт средней
ec7f1a2 Самый тёплый день
d38c30d Первая версия дневника погоды
$ git checkout ec7f1a2 -- weather.py
$ git commit -m "Вернуть прежний подсчёт средней"
[main 07e5848] Вернуть прежний подсчёт средней
 1 file changed, 4 insertions(+), 1 deletion(-)
Задача 6: ок


## Уровень 2. На сочетание идей

### Задача 7. requirements.txt по импортам

Напишите функцию `requirements_for(files, freeze)`, которая составляет `requirements.txt` для проекта.
`files` — словарь «путь → текст файла», как на паре про модули. `freeze` — то, что напечатал
`pip freeze` в окружении, где проект работает. В окружении библиотек обычно больше, чем проект
импортирует сам: например, `requests` при установке тянет за собой `urllib3` и `idna`. В
`requirements.txt` нужны только те библиотеки, которые импортирует проект.

Правила:

1. Смотрим все импорты во всех файлах `.py`, и в начале файла, и внутри функций. Из `import a.b.c`
   и `from a.b import c` нужно только первое имя, `a`.
2. Относительные импорты (`from . import x`, `from .storage import y`) пропускаем: это свой пакет.
3. Модули стандартной библиотеки пропускаем. Их список — `sys.stdlib_module_names`.
4. Свои модули проекта пропускаем: это первые части путей файлов, `weather/stats.py` → `weather`,
   `main.py` → `main`.
5. Имя модуля для `import` и имя библиотеки в PyPI иногда различаются: `import yaml` — это библиотека
   `PyYAML`. Такие пары даны в словаре `IMPORT_TO_PACKAGE`, у остальных библиотек имена совпадают.
6. Версию берём из `freeze`: строки там вида `имя==версия`, пустые строки и строки с `#` в начале
   пропускаем. Имена сравниваем без учёта регистра, считая `-`, `_` и `.` одним и тем же знаком:
   `Scikit_Learn` и `scikit-learn` — одна библиотека. В ответ имя пишем так, как оно записано в `freeze`.
7. Ответ — строки `имя==версия`, каждая с переводом строки в конце, отсортированные по имени с теми же
   поблажками (проще всего сортировать по приведённому имени). Если сторонних импортов нет — пустая строка.
8. Если нужной библиотеки нет во `freeze`, бросьте `ValueError` с именем библиотеки в сообщении.

*Подсказка:* импорты удобно искать модулем `ast`, который разбирает текст программы в дерево:

```python
import ast

tree = ast.parse("import numpy as np\nfrom PIL import Image\nfrom . import stats")
for node in ast.walk(tree):              # обходит всё дерево, в том числе тела функций
    if isinstance(node, ast.Import):
        print([alias.name for alias in node.names])     # ['numpy']
    elif isinstance(node, ast.ImportFrom):
        print(node.module, node.level)                  # PIL 0, потом None 1
```

Привести имя к общему виду можно так: `re.sub(r"[-_.]+", "-", name).lower()`.

In [ ]:
import ast
import re
import sys

IMPORT_TO_PACKAGE = {"yaml": "PyYAML", "PIL": "Pillow", "sklearn": "scikit-learn",
                     "bs4": "beautifulsoup4", "dateutil": "python-dateutil", "cv2": "opencv-python"}


def requirements_for(files: dict[str, str], freeze: str) -> str:
    """Текст requirements.txt: сторонние библиотеки, которые импортирует проект, с версиями из freeze."""
    canon = lambda n: re.sub(r"[-_.]+", "-", n).lower()
    own = {p.split("/")[0].removesuffix(".py") for p in files if p.endswith(".py")}
    imports = set()
    for path, text in files.items():
        if path.endswith(".py"):
            for m in re.finditer(r"(?m)^\s*(?:from\s+(\w+)\S*\s+import|import\s+([\w.]+))", text):
                imports.add((m[1] or m[2]).split(".")[0])
    index = {canon(l.split("==")[0]): l for l in map(str.strip, freeze.splitlines()) if l and l[0] != "#"}
    packages = sorted({IMPORT_TO_PACKAGE.get(m, m) for m in imports - own - sys.stdlib_module_names}, key=canon)
    if missing := [p for p in packages if canon(p) not in index]:
        raise ValueError(missing[0])
    return "".join(index[canon(p)] + "\n" for p in packages)


# ---------- проверка ----------
project = {
    "main.py": '''
import sys
from pathlib import Path

import requests
import yaml

from weather import stats
from weather.storage import read_days
''',
    "weather/__init__.py": '"""Дневник погоды."""\n',
    "weather/stats.py": "import statistics\nfrom collections import Counter\n\nimport numpy as np\n",
    "weather/storage.py": "import json\nimport os.path\n\nfrom dateutil import parser\n\nfrom . import stats\n",
    "weather/plot.py": '''
def draw(days):
    from PIL import Image, ImageDraw
    import tabulate
''',
    "README.md": "Для запуска: import requests, pandas\n",
}
freeze = '''
certifi==2024.8.30
charset-normalizer==3.4.0
idna==3.10
numpy==2.1.3
pillow==11.0.0
python-dateutil==2.9.0.post0
PyYAML==6.0.2
requests==2.32.3
six==1.16.0
tabulate==0.9.0
urllib3==2.2.3
'''
before = dict(project)
answer = requirements_for(project, freeze)
expected = ("numpy==2.1.3\npillow==11.0.0\npython-dateutil==2.9.0.post0\nPyYAML==6.0.2\n"
            "requests==2.32.3\ntabulate==0.9.0\n")
assert answer == expected, f"для проекта погоды получилось:\n{answer}\nа нужно:\n{expected}"
assert project == before, "словарь files менять нельзя"

only_stdlib = {"main.py": "from __future__ import annotations\nimport xml.etree.ElementTree as ET\n"
                          "import importlib.metadata\nfrom collections import Counter\n"}
assert requirements_for(only_stdlib, freeze) == "", "стандартная библиотека в requirements.txt не попадает"
own_tabulate = {"main.py": "import tabulate\n", "tabulate.py": "def tabulate(rows):\n    return str(rows)\n"}
assert requirements_for(own_tabulate, freeze) == "", "tabulate.py лежит в проекте — это свой модуль, а не библиотека"
model = {"model.py": "from sklearn.linear_model import LinearRegression\nimport numpy\nimport numpy as np\n"}
answer = requirements_for(model, "# окружение модели\n\nScikit_Learn==1.5.2\nnumpy==2.1.3\n")
assert answer == "numpy==2.1.3\nScikit_Learn==1.5.2\n", f"имена сравниваются без учёта регистра и - _ . , а получилось:\n{answer}"
error = error_of(requirements_for, {"parse.py": "from bs4 import BeautifulSoup\n"}, freeze)
assert isinstance(error, ValueError), f"если библиотеки нет во freeze, нужен ValueError, а получилось: {error!r}"
assert "beautifulsoup4" in str(error), f"в сообщении нужно имя библиотеки, а оно такое: {error}"
print("Задача 7: ок")

Задача 7: ок


### Задача 8. Две правки в одном файле

В рабочей папке в `weather.py` сразу две правки: в `warmest_day` исправлена ошибка (сравнение `<` искало
самый холодный день) и добавлена новая функция `temp_range`. Сделайте из них два коммита: первый —
только исправление, с сообщением `Исправить поиск самого тёплого дня`, второй — новая функция,
с сообщением `Размах температур`. В конце в рабочей папке должна остаться полная версия файла,
а `git status` должен быть пустым.

*Подсказка:* `git add` кладёт в индекс файл целиком, со всеми правками. Значит, для первого коммита нужно
временно записать в файл версию только с исправлением. Сначала сохраните полную версию:
`full = read("weather.py")`. Последняя закоммиченная версия — `git_output("show HEAD:weather.py")`;
исправьте в ней одну строку через `replace` и запишите файл через `write`. После первого коммита
верните `full`.

In [ ]:
start(8)
# ваш код здесь


# ---------- проверка ----------
history = log()
assert history[:2] == setup_log(), "старые коммиты должны остаться как были"
messages = [message for _, message in history[2:]]
assert messages == ["Исправить поиск самого тёплого дня", "Размах температур"], f"новые коммиты: {messages}"
assert changed("HEAD~1") == {"weather.py"} and changed("HEAD") == {"weather.py"}, "оба коммита меняют только weather.py"
days = [("2026-11-01", 6, "дождь"), ("2026-11-02", 9, "солнце"), ("2026-11-03", 2, "снег")]
first = module_from(file_at("HEAD~1", "weather.py"))
assert first["warmest_day"](days) == "2026-11-02", "в первом коммите warmest_day должна быть исправлена"
assert "temp_range" not in first, "в первом коммите не должно быть temp_range — это второе изменение"
assert file_at("HEAD", "weather.py") == prepared("weather.py"), "во втором коммите должна быть полная версия weather.py — такая, как в начале задачи"
assert status() == [], f"всё должно быть закоммичено, а git status показывает: {status()}"
print("Задача 8: ок")

### Задача 9. Где сломалось

Сейчас `average_temp` в проекте ошибается: для двух дней с температурами 5 и 4 градуса она возвращает
не 4.5. В первом коммите функция работала. Коммитов двенадцать, проверять каждый вручную долго —
поручим это программе.

Напишите функцию `first_bad_commit()`, которая возвращает сообщение первого коммита, в котором
`average_temp` для таких двух дней даёт не 4.5 или падает с ошибкой. Считаем, что после первой поломки
функцию больше не чинили. Проверка вызовет вашу функцию на трёх разных историях, поэтому ответ нужно
вычислять, а не записывать готовым.

*Подсказка:* хеши коммитов от старых к новым — `git_output("log --reverse --format=%H").split()`. Текст
файла из коммита — `git_output(f"show {commit}:weather.py")`, переключаться на коммит не нужно.
Превратить текст в функцию можно так: `namespace = {}`, `exec(text, namespace)`, потом
`namespace["average_temp"](days)`. Сообщение коммита — `git_output(f"log -1 --format=%s {commit}").strip()`.

In [ ]:
start(9)
# посмотреть историю: git("log --oneline")


def first_bad_commit() -> str:
    """Сообщение первого коммита, где average_temp для дней с 5 и 4 градусами даёт не 4.5."""
    # ваш код здесь


# ---------- проверка ----------
for variant, expected in [(0, "Ускорить подсчёт средней"), (1, "Переименовать переменные в average_temp"),
                          (2, "Аннотации типов")]:
    start(9, variant, quiet=True)
    answer = first_bad_commit()
    assert answer == expected, f"история {variant}: ответ {answer!r}, а должен быть {expected!r}"
print("Задача 9: ок")

### Задача 10. Отменить коммит, не переписывая историю

Когда-то в проект добавили архив записей за октябрь, а потом коммитом «Убрать архив» удалили его вместе
с упоминанием в `README.md`. Оказалось, зря. Отмените этот коммит целиком: файл архива и строка в README
должны вернуться, а всё, что сделано после (новые записи и функция `warmest_day`), должно остаться.
Историю не переписывайте: все шесть коммитов остаются на месте, отмена — новым коммитом.

*Подсказка:* найдите коммит в `git log --oneline`. Какая команда из блока 5 создаёт коммит, обратный
указанному?

In [ ]:
start(10)
# ваш код здесь


# ---------- проверка ----------
history, original = log(), setup_log()
assert history[:6] == original, "все шесть старых коммитов должны остаться как были"
assert len(history) == 7, f"нужен ровно один новый коммит, а их {len(history) - 6}"
archive = original[1][0]                        # коммит «Архив за октябрь»
assert file_at("HEAD", "archive_october.txt") == file_at(archive, "archive_october.txt"), "архив за октябрь не вернулся"
assert file_at("HEAD", "README.md") == file_at(archive, "README.md"), "README.md должен снова упоминать архив"
last = original[5][0]
assert file_at("HEAD", "days.txt") == file_at(last, "days.txt"), "записи за 6–8 ноября потерялись"
assert file_at("HEAD", "weather.py") == file_at(last, "weather.py"), "функция warmest_day потерялась"
assert changed("HEAD") == {"archive_october.txt", "README.md"}, f"новый коммит должен менять архив и README, а меняет: {sorted(changed('HEAD'))}"
assert status() == [], f"всё должно быть закоммичено, а git status показывает: {status()}"
print("Задача 10: ок")

## Уровень 3. Со звёздочкой

### Задача 11. Как Git хранит коммиты ★

Git хранит всё в папке `.git/objects` в виде **объектов**. Объект — это байты вида
`тип размер\0содержимое`, сжатые модулем zlib; размер — длина содержимого в байтах, `\0` — нулевой байт.
Имя объекта — SHA-1 хеш этих байтов до сжатия. Первые две цифры хеша — имя папки, остальные 38 — имя
файла: объект `5d41402a…` лежит в `.git/objects/5d/41402a…`. Содержимое каждого файла проекта хранится
объектом типа `blob`, коммит — объектом типа `commit`. Распакованный коммит выглядит так:

```
commit 251\0tree 9c4f…
parent 3a1b…
author Ученик 9С <student@example.com> 1793862000 +0300
committer Ученик 9С <student@example.com> 1793862000 +0300

Дождливые дни

Функция rainy_days возвращает даты.
```

В строке `parent` — хеш предыдущего коммита, у первого коммита её нет. После пустой строки идёт сообщение,
его первую строку показывает `git log --oneline`. Где «мы сейчас», записано в файле `.git/HEAD`: строка
`ref: refs/heads/main` значит «смотри файл `.git/refs/heads/main`», а в нём хранится хеш последнего коммита
ветки.

Напишите две функции, **не запуская git** (проверка это запретит):

- `blob_hash(content)` — хеш, под которым Git сохранит файл с содержимым `content` (это байты);
- `history(repo)` — коммиты ветки, на которую указывает `HEAD`, от нового к старому: список пар
  (первые 7 цифр хеша, первая строка сообщения). `repo` — путь к папке проекта.

*Подсказка:* `hashlib.sha1(data).hexdigest()`, `zlib.decompress(data)`, `Path(...).read_bytes()`. Байты
делятся почти как строки: `data.split(b"\0", 1)`, а в строку превращаются методом `.decode("utf-8")`.

In [ ]:
import hashlib
import zlib
from pathlib import Path

start(11)


def blob_hash(content: bytes) -> str:
    """Хеш объекта blob для файла с содержимым content — как у git hash-object."""
    # ваш код здесь


def history(repo: str) -> list[tuple[str, str]]:
    """Коммиты ветки HEAD от нового к старому: (7 цифр хеша, первая строка сообщения)."""
    # ваш код здесь


# ---------- проверка ----------
samples = [b"", b"hello\n", "Дневник погоды\n".encode(), bytes(range(256))]
expected_hashes = [git_hash(sample) for sample in samples]
with no_programs():
    answer = [blob_hash(sample) for sample in samples]
assert answer == expected_hashes, f"blob_hash: {answer}\nа git hash-object даёт: {expected_hashes}"

for variant in (0, 1):
    start(11, variant, quiet=True)
    expected = [(commit[:7], message) for commit, message in reversed(log())]
    with no_programs():
        answer = history(str(ROOT))
    assert answer == expected, f"история {variant}: {answer}\nа git log даёт: {expected}"
print("Задача 11: ок")

### Задача 12. Свой .gitignore ★

Напишите функцию `is_ignored(path, patterns)`, которая решает, игнорирует ли Git файл `path`, если
в `.gitignore` записаны строки `patterns`. `path` — путь к файлу от корня проекта через `/`, например
`weather/__pycache__/stats.pyc`; все части пути, кроме последней, — папки. Нужно поддержать такие правила:

1. Пустые строки и строки, которые начинаются с `#`, пропускаются.
2. `!` в начале — исключение: подходящий путь, наоборот, не игнорируется.
3. `/` в конце значит, что шаблон подходит только к папкам. При сравнении этот `/` отбрасывается.
4. Если в шаблоне (без `/` в конце) есть `/` в начале или в середине, он отсчитывается от корня, как
   `/build` или `docs/*.html`: начальный `/` отбрасывается, и шаблон сравнивается со всем путём от корня.
   Иначе шаблон сравнивается только с последним именем пути и подходит на любой глубине.
5. В шаблоне бывают `*` (любые символы, кроме `/`), `?` (один любой символ, кроме `/`) и `[...]` (один
   символ из перечисленных).
6. Если к пути подходит несколько шаблонов, решает последний из них.
7. Если игнорируется папка, игнорируется всё внутри неё, и `!` для файла внутри уже не помогает. Поэтому
   сначала проверьте каждую папку на пути, начиная с верхней, и только потом сам файл.

Проверка сравнит ответы функции с ответами настоящего Git на нескольких наборах шаблонов.

*Подсказка:* `fnmatch.fnmatchcase(имя, шаблон)` понимает `*`, `?` и `[...]`, но её `*` захватывает и `/`.
Поэтому сравнивайте по частям: разделите и путь, и шаблон по `/` и сравнивайте часть с частью. Шаблон
из правила 4 подходит, только если частей в нём столько же, сколько у проверяемого пути.

In [ ]:
from fnmatch import fnmatchcase


def is_ignored(path: str, patterns: list[str]) -> bool:
    """Игнорирует ли Git файл path, если в .gitignore записаны строки patterns."""
    # ваш код здесь


# ---------- проверка ----------
cases = [
    (["*.log", "!keep.log"],
     ["a.log", "keep.log", "logs/x.log", "logs/keep.log", "notes.txt", "log.txt"]),
    (["build/", "/dist", "docs/*.html"],
     ["build/app.py", "src/build/app.py", "build", "dist/x.js", "src/dist/x.js", "docs/index.html",
      "docs/api/ref.html", "index.html"]),
    (["__pycache__/", ".venv/", "secret?.txt", "data/[0-9]*.csv"],
     ["__pycache__/m.pyc", "pkg/__pycache__/m.pyc", ".venv/bin/python", "pkg/.venv/x", "secret1.txt",
      "secret12.txt", "dir/secret2.txt", "data/2026.csv", "data/raw.csv", "data/old/2025.csv"]),
    (["# временные файлы", "", "temp", "!temp/keep.txt", "logs/", "!logs/today.log"],
     ["temp", "temp/a.txt", "temp/keep.txt", "src/temp", "logs/today.log", "logs.txt", "app/logs/x.txt"]),
    (["out/*", "!out/keep.txt", "a/b"],
     ["out/a.txt", "out/keep.txt", "out/sub/b.txt", "out", "a/b", "a/b/c.txt", "x/a/b"]),
    (["*", "!*.py", "!*/"],
     ["main.py", "README.md", "pkg/mod.py", "pkg/data.csv", "pkg/sub/deep.py"]),
]
for patterns, paths in cases:
    expected = git_ignores(patterns, paths)          # спрашиваем настоящий Git
    for path in paths:
        answer = is_ignored(path, patterns)
        assert answer == (path in expected), f"шаблоны {patterns}, путь {path!r}: Git отвечает {path in expected}, а функция — {answer}"
print("Задача 12: ок")